# Multi-Entity Biomedical NER — Experiment Runner

One-click notebook to launch every training configuration (1..6) with one
or more seeds, then collate the per-seed evaluation JSONs into mean ± std
tables.

This notebook calls the training functions **directly** in the kernel's
Python process — no `bash`, no `subprocess`, no environment crossing. Make
sure the kernel uses the Python where `requirements.txt` is installed.

Bash scripts under `scripts/` remain available for headless multi-seed
runs on Vast.ai / SSH sessions; the notebook does not invoke them.

## 1. Setup — run once per session

In [1]:
import os
import sys
from pathlib import Path

# Ensure we work from the project root so relative paths in YAML resolve.
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f'Project root: {PROJECT_ROOT}')
print(f'Python executable: {sys.executable}')

import torch, transformers
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Transformers: {transformers.__version__}')

from src.config import load_config
from src.training.trainer_single import train_single
from src.training.trainer_sequential import train_sequential
from src.training.trainer_joint import train_joint
from src.utils.aggregate import aggregate_seeds
from src.utils.logging import configure_logging

# One log file per notebook session.
configure_logging(PROJECT_ROOT / 'outputs' / 'logs' / 'notebook', run_name='session')

Project root: c:\Users\Vito\Semester_4 - Copy\FP_NLP\BioNER-DS
Python executable: c:\Users\Vito\Semester_4 - Copy\FP_NLP\.venv\Scripts\python.exe


c:\Users\Vito\Semester_4 - Copy\FP_NLP\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch: 2.14.0.dev20260618+cu130
CUDA available: True
GPU: NVIDIA GeForce RTX 5060 Laptop GPU
Transformers: 4.57.6


WindowsPath('c:/Users/Vito/Semester_4 - Copy/FP_NLP/BioNER-DS/outputs/logs/notebook/run_20260620_171924_session.log')

## 2. (Optional) Install requirements

Skip if your venv already has the deps. PyTorch should be installed
separately first (CPU or matching CUDA build) before running this cell.

In [ ]:
%pip install -r requirements.txt

## 3. Verify dataset files exist

In [2]:
for rel in [
    'dataset/bc5cdr/bc5cdr_train.jsonl',
    'dataset/bc5cdr/bc5cdr_validation.jsonl',
    'dataset/bc5cdr/bc5cdr_test.jsonl',
    'dataset/pubmed/pubmed_scrapping.jsonl',
    'dataset/pubmed/pubmed_test.jsonl',
]:
    p = Path(rel)
    print(f"  {rel}: {'OK' if p.exists() else 'MISSING'}  ({p.stat().st_size if p.exists() else 0:,} bytes)")

  dataset/bc5cdr/bc5cdr_train.jsonl: OK  (3,984,432 bytes)
  dataset/bc5cdr/bc5cdr_validation.jsonl: OK  (3,982,519 bytes)
  dataset/bc5cdr/bc5cdr_test.jsonl: OK  (4,234,325 bytes)
  dataset/pubmed/pubmed_scrapping.jsonl: OK  (40,332,751 bytes)
  dataset/pubmed/pubmed_test.jsonl: OK  (101,668 bytes)


## 4. Pick seeds and helper

Default is three seeds for paper-grade mean ± std reporting. Change `SEEDS`
once here and every config cell below will use the new value.

Set `SMOKE_TEST = True` for a quick plumbing check (16 training sentences,
1 epoch). Set it back to `False` for full runs.

In [3]:
SEEDS = [42, 1337, 2024]
SMOKE_TEST = False  # flip to True for a quick pipeline check

TRAINER_BY_STRATEGY = {
    'single': train_single,
    'sequential': train_sequential,
    'joint_uniform': train_joint,
    'joint_noise_aware': train_joint,
}

def run_config(config_path: str, seeds=None, smoke_test=None) -> None:
    """Train one experiment across the given seeds, then aggregate."""
    seeds = seeds if seeds is not None else SEEDS
    smoke_test = smoke_test if smoke_test is not None else SMOKE_TEST
    cfg = load_config(config_path)
    trainer_fn = TRAINER_BY_STRATEGY[cfg.training.strategy]
    print('=' * 70)
    print(f'  {cfg.experiment.name}  |  strategy={cfg.training.strategy}  |  seeds={seeds}')
    print('=' * 70)
    for seed in seeds:
        print(f'\n>>> {cfg.experiment.name} — seed {seed}\n')
        trainer_fn(config=cfg, seed=seed, smoke_test=smoke_test)
    print(f'\n>>> Aggregating {cfg.experiment.name}')
    aggregate_seeds(cfg.experiment.name, base_dir=str(PROJECT_ROOT / 'outputs'))

print(f'Using SEEDS={SEEDS}, SMOKE_TEST={SMOKE_TEST}')

Using SEEDS=[42, 1337, 2024], SMOKE_TEST=False


## 5. Config 1 — BERT-base + BC5CDR (lower-bound baseline)

In [ ]:
run_config('configs/config_1_bert_base.yaml')

## 6. Config 2 — BioBERT + BC5CDR

In [ ]:
run_config('configs/config_2_biobert.yaml')

## 7. Config 3 — PubMedBERT + BC5CDR

In [ ]:
run_config('configs/config_3_pubmedbert.yaml')

## 8. Config 4 — Sequential (BC5CDR → silver)

Trains 9-tag head; phase 1 sees only Chem/Dis, phase 2 lifts Virus/Gene
from the silver corpus. Forgetting score on Chem/Dis is logged automatically.

In [ ]:
run_config('configs/config_4_sequential.yaml')

## 9. Config 5 — Joint training (uniform sample weights)

In [ ]:
run_config('configs/config_5_joint_uniform.yaml')

## 10. Config 6 — Joint training (noise-aware, silver=0.3)

In [7]:
run_config('configs/config_6_joint_noise_aware.yaml')

  config_6_joint_noise_aware  |  strategy=joint_noise_aware  |  seeds=[42, 1337, 2024]

>>> config_6_joint_noise_aware — seed 42

2026-06-20 19:47:12 |     INFO | src.models.ner_model | Loading tokenizer for backbone=microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext
2026-06-20 19:47:14 |     INFO | src.models.ner_model | Building token-classification model from microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext with num_labels=9


Some weights of BertForTokenClassification were not initialized from the model checkpoint at microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


2026-06-20 19:47:16 |     INFO | src.data.dataset | Loaded 5119 sentences (109077 tokens) from bc5cdr_train.jsonl with label_field='decoded_tags_pseudo_final', source='gold', sample_weight=1.000.
2026-06-20 19:47:16 |     INFO | src.data.dataset | Top-5 labels in bc5cdr_train.jsonl: O=96239, B-Chemical=5203, B-Disease=4182, I-Disease=2570, I-Chemical=571
2026-06-20 19:47:44 |     INFO | src.data.dataset | Loaded 40946 sentences (1095014 tokens) from pubmed_scrapping.jsonl with label_field='decoded_tags_pseudo_final', source='silver', sample_weight=0.500.
2026-06-20 19:47:44 |     INFO | src.data.dataset | Top-5 labels in pubmed_scrapping.jsonl: O=1048986, B-Virus=21469, B-Gene=13449, I-Virus=11073, I-Gene=33
2026-06-20 19:47:44 |     INFO | src.data.loaders | Concatenated 2 datasets with sizes [5119, 40946] for joint training.
2026-06-20 19:47:44 |     INFO | src.data.dataset | Loaded 5218 sentences (108713 tokens) from bc5cdr_validation.jsonl with label_field='decoded_tags_pseudo_fina

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Chemical Precision,Chemical Recall,Chemical F1,Chemical Support,Disease Precision,Disease Recall,Disease F1,Disease Support,Gene Precision,Gene Recall,Gene F1,Gene Support,Virus Precision,Virus Recall,Virus F1,Virus Support
1,0.056200,0.232444,0.800000,0.459858,0.584013,0.852377,0.415747,0.558894,5347.000000,0.766484,0.525919,0.623812,4244.000000,0.692982,0.282143,0.401015,280.000000,0.274510,0.736842,0.400000,19.000000
2,0.050200,0.167173,0.713663,0.728817,0.721161,0.884189,0.805311,0.842909,5347.000000,0.553688,0.659755,0.602086,4244.000000,0.679104,0.325000,0.439614,280.000000,0.282051,0.578947,0.379310,19.000000
3,0.032900,0.139703,0.799137,0.786451,0.792743,0.872345,0.844773,0.858337,5347.000000,0.719462,0.743874,0.731464,4244.000000,0.760331,0.328571,0.458853,280.000000,0.260870,0.631579,0.369231,19.000000
4,0.018300,0.131036,0.847250,0.811527,0.829004,0.902581,0.869834,0.885905,5347.000000,0.798254,0.754241,0.775624,4244.000000,0.593407,0.578571,0.585895,280.000000,0.324324,0.631579,0.428571,19.000000
5,0.010300,0.135070,0.852596,0.815268,0.833514,0.913249,0.866280,0.889145,5347.000000,0.801673,0.767672,0.784304,4244.000000,0.571930,0.582143,0.576991,280.000000,0.277778,0.526316,0.363636,19.000000


2026-06-20 21:31:29 |     INFO | src.data.dataset | Loaded 5728 sentences (116015 tokens) from bc5cdr_test.jsonl with label_field='decoded_tags', source='gold', sample_weight=1.000.
2026-06-20 21:31:29 |     INFO | src.data.dataset | Top-5 labels in bc5cdr_test.jsonl: O=103381, B-Chemical=5385, B-Disease=4424, I-Disease=2424, I-Chemical=401
2026-06-20 21:31:29 |     INFO | src.data.dataset | Loaded 100 sentences (2766 tokens) from pubmed_test.jsonl with label_field='decoded_tags_pseudo_final', source='gold', sample_weight=1.000.
2026-06-20 21:31:29 |     INFO | src.data.dataset | Top-5 labels in pubmed_test.jsonl: O=2544, B-Virus=102, B-Gene=63, I-Virus=53, B-Chemical=4
2026-06-20 21:31:29 |     INFO | src.training.trainer_base | Evaluating on test set 'test_bc5cdr' (5728 examples)


...

2026-06-20 22:13:23 |     INFO | src.data.dataset | Loaded 5119 sentences (109077 tokens) from bc5cdr_train.jsonl with label_field='decoded_tags_pseudo_final', source='gold', sample_weight=1.000.
2026-06-20 22:13:23 |     INFO | src.data.dataset | Top-5 labels in bc5cdr_train.jsonl: O=96239, B-Chemical=5203, B-Disease=4182, I-Disease=2570, I-Chemical=571
2026-06-20 22:13:29 |     INFO | src.data.dataset | Loaded 40946 sentences (1095014 tokens) from pubmed_scrapping.jsonl with label_field='decoded_tags_pseudo_final', source='silver', sample_weight=0.500.
2026-06-20 22:13:29 |     INFO | src.data.dataset | Top-5 labels in pubmed_scrapping.jsonl: O=1048986, B-Virus=21469, B-Gene=13449, I-Virus=11073, I-Gene=33
2026-06-20 22:13:29 |     INFO | src.data.loaders | Concatenated 2 datasets with sizes [5119, 40946] for joint training.
2026-06-20 22:13:30 |     INFO | src.data.dataset | Loaded 5218 sentences (108713 tokens) from bc5cdr_validation.jsonl with label_field='decoded_tags_pseudo_fina

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Chemical Precision,Chemical Recall,Chemical F1,Chemical Support,Disease Precision,Disease Recall,Disease F1,Disease Support,Gene Precision,Gene Recall,Gene F1,Gene Support,Virus Precision,Virus Recall,Virus F1,Virus Support
1,0.290500,0.741689,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,5347.000000,0.000000,0.000000,0.000000,4244.000000,0.000000,0.000000,0.000000,280.000000,0.000000,0.000000,0.000000,19.000000
2,0.294500,0.699865,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,5347.000000,0.000000,0.000000,0.000000,4244.000000,0.000000,0.000000,0.000000,280.000000,0.000000,0.000000,0.000000,19.000000
3,0.278400,0.717662,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,5347.000000,0.000000,0.000000,0.000000,4244.000000,0.000000,0.000000,0.000000,280.000000,0.000000,0.000000,0.000000,19.000000
4,0.282100,0.728846,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,5347.000000,0.000000,0.000000,0.000000,4244.000000,0.000000,0.000000,0.000000,280.000000,0.000000,0.000000,0.000000,19.000000
5,0.292000,0.727133,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,5347.000000,0.000000,0.000000,0.000000,4244.000000,0.000000,0.000000,0.000000,280.000000,0.000000,0.000000,0.000000,19.000000


2026-06-21 01:04:27 |     INFO | src.data.dataset | Loaded 5728 sentences (116015 tokens) from bc5cdr_test.jsonl with label_field='decoded_tags', source='gold', sample_weight=1.000.
2026-06-21 01:04:27 |     INFO | src.data.dataset | Top-5 labels in bc5cdr_test.jsonl: O=103381, B-Chemical=5385, B-Disease=4424, I-Disease=2424, I-Chemical=401
2026-06-21 01:04:27 |     INFO | src.data.dataset | Loaded 100 sentences (2766 tokens) from pubmed_test.jsonl with label_field='decoded_tags_pseudo_final', source='gold', sample_weight=1.000.
2026-06-21 01:04:27 |     INFO | src.data.dataset | Top-5 labels in pubmed_test.jsonl: O=2544, B-Virus=102, B-Gene=63, I-Virus=53, B-Chemical=4
2026-06-21 01:04:27 |     INFO | src.training.trainer_base | Evaluating on test set 'test_bc5cdr' (5728 examples)


2026-06-21 01:07:27 |     INFO | src.evaluation.metrics | Saved evaluation results to outputs\results\config_6_joint_noise_aware\seed_2024\eval_test_bc5cdr.json
2026-06-21 01:07:27 |     INFO | src.training.trainer_base | Test 'test_bc5cdr' overall: precision=0.0000, recall=0.0000, f1=0.0000
2026-06-21 01:07:27 |     INFO | src.training.trainer_base | Test 'test_bc5cdr' entity Chemical: f1=0.0000, p=0.0000, r=0.0000, support=5385
2026-06-21 01:07:27 |     INFO | src.training.trainer_base | Test 'test_bc5cdr' entity Disease: f1=0.0000, p=0.0000, r=0.0000, support=4424
2026-06-21 01:07:27 |     INFO | src.training.trainer_base | Evaluating on test set 'test_pubmed' (100 examples)


2026-06-21 01:07:41 |     INFO | src.evaluation.metrics | Saved evaluation results to outputs\results\config_6_joint_noise_aware\seed_2024\eval_test_pubmed.json
2026-06-21 01:07:41 |     INFO | src.training.trainer_base | Test 'test_pubmed' overall: precision=0.0000, recall=0.0000, f1=0.0000
2026-06-21 01:07:41 |     INFO | src.training.trainer_base | Test 'test_pubmed' entity Chemical: f1=0.0000, p=0.0000, r=0.0000, support=4
2026-06-21 01:07:41 |     INFO | src.training.trainer_base | Test 'test_pubmed' entity Gene: f1=0.0000, p=0.0000, r=0.0000, support=63
2026-06-21 01:07:41 |     INFO | src.training.trainer_base | Test 'test_pubmed' entity Virus: f1=0.0000, p=0.0000, r=0.0000, support=102

>>> Aggregating config_6_joint_noise_aware
2026-06-21 01:07:41 |     INFO | src.utils.aggregate | Wrote aggregated JSON to c:\Users\Vito\Semester_4 - Copy\FP_NLP\BioNER-DS\outputs\results\config_6_joint_noise_aware\aggregated_results.json
2026-06-21 01:07:41 |     INFO | src.utils.aggregate | Wr

## 11. View aggregated results

Each `run_config(...)` call already wrote `aggregated_results.{json,md}`
into `outputs/results/<config>/`. The cell below renders every available
Markdown table for quick side-by-side comparison.

In [8]:
from IPython.display import Markdown, display
for results_dir in sorted(Path('outputs/results').glob('config_*')):
    md_file = results_dir / 'aggregated_results.md'
    if md_file.exists():
        display(Markdown(md_file.read_text(encoding='utf-8')))
    else:
        print(f'[no aggregated_results.md yet] {results_dir.name}')

# Config: config_1_bert_base

## Test Set: test_bc5cdr

Aggregated over 3 seeds: [1337, 2024, 42]

| Metric | Mean ± Std |
|---|---|
| Overall Precision | 0.8385 ± 0.0066 |
| Overall Recall | 0.8135 ± 0.0039 |
| Overall F1 | 0.8258 ± 0.0045 |

### Per-Entity

| Entity | Precision | Recall | F1 | Support |
|---|---|---|---|---|
| Chemical | 0.895 ± 0.005 | 0.842 ± 0.010 | 0.868 ± 0.003 | 5385 |
| Disease | 0.775 ± 0.016 | 0.779 ± 0.003 | 0.777 ± 0.007 | 4424 |


# Config: config_2_biobert

## Test Set: test_bc5cdr

Aggregated over 3 seeds: [1337, 2024, 42]

| Metric | Mean ± Std |
|---|---|
| Overall Precision | 0.8703 ± 0.0046 |
| Overall Recall | 0.8807 ± 0.0010 |
| Overall F1 | 0.8755 ± 0.0020 |

### Per-Entity

| Entity | Precision | Recall | F1 | Support |
|---|---|---|---|---|
| Chemical | 0.916 ± 0.005 | 0.916 ± 0.004 | 0.916 ± 0.002 | 5385 |
| Disease | 0.817 ± 0.013 | 0.838 ± 0.004 | 0.827 ± 0.006 | 4424 |


# Config: config_3_pubmedbert

## Test Set: test_bc5cdr

Aggregated over 3 seeds: [1337, 2024, 42]

| Metric | Mean ± Std |
|---|---|
| Overall Precision | 0.8924 ± 0.0036 |
| Overall Recall | 0.9074 ± 0.0034 |
| Overall F1 | 0.8998 ± 0.0013 |

### Per-Entity

| Entity | Precision | Recall | F1 | Support |
|---|---|---|---|---|
| Chemical | 0.936 ± 0.005 | 0.938 ± 0.007 | 0.937 ± 0.002 | 5385 |
| Disease | 0.841 ± 0.005 | 0.870 ± 0.004 | 0.855 ± 0.004 | 4424 |


[no aggregated_results.md yet] config_4_sequential


# Config: config_5_joint_uniform

## Test Set: test_bc5cdr

Aggregated over 3 seeds: [1337, 2024, 42]

| Metric | Mean ± Std |
|---|---|
| Overall Precision | 0.8303 ± 0.0091 |
| Overall Recall | 0.8220 ± 0.0053 |
| Overall F1 | 0.8261 ± 0.0071 |

### Per-Entity

| Entity | Precision | Recall | F1 | Support |
|---|---|---|---|---|
| Chemical | 0.927 ± 0.003 | 0.853 ± 0.008 | 0.888 ± 0.003 | 5385 |
| Disease | 0.811 ± 0.019 | 0.785 ± 0.002 | 0.798 ± 0.010 | 4424 |

## Test Set: test_pubmed

Aggregated over 3 seeds: [1337, 2024, 42]

| Metric | Mean ± Std |
|---|---|
| Overall Precision | 0.9684 ± 0.0034 |
| Overall Recall | 0.9665 ± 0.0034 |
| Overall F1 | 0.9674 ± 0.0030 |

### Per-Entity

| Entity | Precision | Recall | F1 | Support |
|---|---|---|---|---|
| Chemical | 0.000 ± 0.000 | 0.000 ± 0.000 | 0.000 ± 0.000 | 4 |
| Gene | 0.922 ± 0.008 | 1.000 ± 0.000 | 0.959 ± 0.004 | 63 |
| Virus | 1.000 ± 0.000 | 0.984 ± 0.006 | 0.992 ± 0.003 | 102 |


# Config: config_6_joint_noise_aware

## Test Set: test_bc5cdr

Aggregated over 3 seeds: [1337, 2024, 42]

| Metric | Mean ± Std |
|---|---|
| Overall Precision | 0.5401 ± 0.4677 |
| Overall Recall | 0.5422 ± 0.4696 |
| Overall F1 | 0.5411 ± 0.4686 |

### Per-Entity

| Entity | Precision | Recall | F1 | Support |
|---|---|---|---|---|
| Chemical | 0.606 ± 0.525 | 0.566 ± 0.490 | 0.585 ± 0.507 | 5385 |
| Disease | 0.521 ± 0.451 | 0.513 ± 0.444 | 0.517 ± 0.448 | 4424 |

## Test Set: test_pubmed

Aggregated over 3 seeds: [1337, 2024, 42]

| Metric | Mean ± Std |
|---|---|
| Overall Precision | 0.6278 ± 0.5437 |
| Overall Recall | 0.6371 ± 0.5518 |
| Overall F1 | 0.6324 ± 0.5477 |

### Per-Entity

| Entity | Precision | Recall | F1 | Support |
|---|---|---|---|---|
| Chemical | N/A | N/A | N/A | 4 |
| Gene | 0.591 ± 0.512 | 0.661 ± 0.573 | 0.624 ± 0.541 | 63 |
| Virus | 0.654 ± 0.566 | 0.647 ± 0.560 | 0.650 ± 0.563 | 102 |

_Entities marked N/A have fewer than 10 gold spans in this test set (Chemical); the raw numbers remain available in `aggregated_results.json`._


## 12. (Optional) Inference smoke test

After Config 4 finishes, run inference on the PubMed test set using the
phase-2 checkpoint. Change the path to whichever checkpoint you want.

In [ ]:
from src.inference.predictor import NERPredictor
import json

CHECKPOINT = 'outputs/checkpoints/config_4_sequential/seed_42/phase2/best_model'
INPUT_JSONL = 'dataset/pubmed/pubmed_test.jsonl'
OUTPUT_JSONL = 'outputs/predictions/config_4_pubmed_test.jsonl'

predictor = NERPredictor(CHECKPOINT, batch_size=16)
records = [json.loads(line) for line in Path(INPUT_JSONL).open('r', encoding='utf-8') if line.strip()]
preds = predictor.predict_records(records, token_field='tokens', id_field='sentence_id')

Path(OUTPUT_JSONL).parent.mkdir(parents=True, exist_ok=True)
with Path(OUTPUT_JSONL).open('w', encoding='utf-8') as fp:
    for record in preds:
        fp.write(json.dumps(record, ensure_ascii=False) + '\n')
print(f'Wrote {len(preds)} predictions to {OUTPUT_JSONL}')
print('First record:', json.dumps(preds[0], indent=2)[:600])